# SASRec on MovieLens 1M — Regressor with MSE Loss

This notebook is a companion to `sasrec_movielens1m_ratings.ipynb` and demonstrates
`SASRecRegressorEstimator` using **MSE loss** on the same MovieLens 1M rating data.

Key differences from the ratings (BCE) notebook:
- **Estimator**: `SASRecRegressorEstimator` instead of `SASRecClassifierEstimator`
- **Loss**: MSE — the model regresses toward the normalised rating value directly
- **Negatives**: `num_negatives=1` with target score 0.0 (below any real rating)

Use this when your outcome is a true continuous variable (revenue, time-spent, etc.).


## 1. Imports

In [1]:
import logging
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.estimator.sequential import SASRecRegressorEstimator
from skrec.recommender.sequential import SequentialRecommender
from skrec.scorer.sequential import SequentialScorer

# Show training loss logs from the estimator
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/sasrec-ratings-mse")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

Imports OK


## 2. Download MovieLens 1M

Data is stored in `examples/movielens-1m/data/raw/` (excluded from git via `.gitignore`).  
If the files already exist from running a previous notebook, this is a no-op.

In [2]:
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
zip_path = RAW_DIR / "ml-1m.zip"

if not (RAW_DIR / "ratings.dat").exists():
    print("Downloading MovieLens 1M...")
    urllib.request.urlretrieve(ML1M_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith(".dat"):
                filename = Path(name).name
                with zf.open(name) as src, open(RAW_DIR / filename, "wb") as dst:
                    dst.write(src.read())
    print("Downloaded and extracted.")
else:
    print("Already downloaded.")

Already downloaded.


## 3. Load, Preprocess, and Split

In [3]:
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
)
movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1",
)

interactions = pd.DataFrame(
    {
        "USER_ID": ratings["UserID"].astype(str),
        "ITEM_ID": ratings["MovieID"].astype(str),
        "OUTCOME": ratings["Rating"].astype(float),
        "TIMESTAMP": ratings["Timestamp"],
    }
)
items = pd.DataFrame({"ITEM_ID": movies["MovieID"].astype(str)})

# Leave-last-two-out (matching original SASRec paper)
interactions = interactions.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)
user_counts = interactions.groupby("USER_ID").size()
valid_users = user_counts[user_counts >= 5].index
interactions = interactions[interactions["USER_ID"].isin(valid_users)].reset_index(drop=True)

interactions["rank"] = interactions.groupby("USER_ID").cumcount(ascending=False)
test_df = interactions[interactions["rank"] == 0].drop(columns=["rank"]).reset_index(drop=True)
valid_df = interactions[interactions["rank"] == 1].drop(columns=["rank"]).reset_index(drop=True)
train_df = interactions[interactions["rank"] >= 2].drop(columns=["rank"]).reset_index(drop=True)

n_users = train_df.USER_ID.nunique()
print(f"Train: {len(train_df):,}  |  Valid: {len(valid_df):,}  |  Test: {len(test_df):,}  |  Users: {n_users:,}")

Train: 988,129  |  Valid: 6,040  |  Test: 6,040  |  Users: 6,040


## 4. Create Datasets

In [4]:
train_path = str(DATA_DIR / "train_interactions.csv")
items_path = str(DATA_DIR / "items.csv")

if not Path(train_path).exists():
    train_df.to_csv(train_path, index=False)
if not Path(items_path).exists():
    items.to_csv(items_path, index=False)

interactions_ds = InteractionsDataset(data_location=train_path)
items_ds = ItemsDataset(data_location=items_path)
print("Datasets created.")

Datasets created.


## 5. Build and Train SASRec — Regressor, `num_negatives=1`

Loss at each position:
```
MSE(predicted_score_for_next_item, actual_rating_of_next_item)
+ MSE(predicted_score_for_negative_item, 0.0)
```
Random unseen items receive `target=0.0`, anchoring their scores below any real
interaction (minimum real target = 1.0). This is the key fix vs. the `num_negatives=0`
variant, which left unseen item scores unconstrained and produced non-personalised results.


In [5]:
estimator = SASRecRegressorEstimator(
    hidden_units=50,
    num_blocks=2,
    num_heads=1,
    dropout_rate=0.2,
    num_negatives=1,  # anchor unseen item scores below the minimum real rating (1.0)
    learning_rate=0.001,
    epochs=200,  # paper-recommended; 50 is insufficient for personalization to emerge
    batch_size=128,
    optimizer_name="adam",
    loss_fn_name="mse",
    verbose=1,
)

scorer = SequentialScorer(estimator)
recommender = SequentialRecommender(scorer, max_len=200)

print("Training SASRec (Regressor, explicit negatives)...")
recommender.train(items_ds=items_ds, interactions_ds=interactions_ds)
print("Training complete.")

Training SASRec (Regressor, explicit negatives)...


2026-04-29 00:59:37,644 - skrec.recommender.sequential.sequential_recommender - WARNING SequentialRecommender.max_len=200 overrides SASRecRegressorEstimator.max_len=50. Pass the same max_len to both, or rely on the recommender's value.


2026-04-29 00:59:37,644 skrec.recommender.sequential.sequential_recommender WARNING SequentialRecommender.max_len=200 overrides SASRecRegressorEstimator.max_len=50. Pass the same max_len to both, or rely on the recommender's value.


2026-04-29 00:59:38,008 - skrec.recommender.sequential.sequential_recommender - INFO Built sequences for 6040 users (max_len=200, has_outcome=True).


2026-04-29 00:59:38,008 skrec.recommender.sequential.sequential_recommender INFO Built sequences for 6040 users (max_len=200, has_outcome=True).


...

2026-04-29 01:28:04,845 - skrec.estimator.sequential.sasrec_estimator - INFO Epoch [199/200], Loss: 2.7704


2026-04-29 01:28:04,845 skrec.estimator.sequential.sasrec_estimator INFO Epoch [199/200], Loss: 2.7704


2026-04-29 01:28:11,734 - skrec.estimator.sequential.sasrec_estimator - INFO Epoch [200/200], Loss: 2.7686


2026-04-29 01:28:11,734 skrec.estimator.sequential.sasrec_estimator INFO Epoch [200/200], Loss: 2.7686


Training complete.


## 6. Evaluate: HR@10 and NDCG@10

In [6]:
rng = np.random.default_rng(42)
all_item_ids = np.array(list(scorer.item_names))

known_items = set(scorer.item_names)
eval_test_df = test_df[test_df["ITEM_ID"].isin(known_items)].copy()
eval_users = set(eval_test_df["USER_ID"])

# For test evaluation: give model all n-1 items (train + valid) as history.
# This matches SASRec paper: test uses full history up to (but not including) the test item.
eval_train_df = train_df[train_df["USER_ID"].isin(eval_users)].copy()
eval_valid_df = valid_df[valid_df["USER_ID"].isin(eval_users)].copy()
eval_history_df = pd.concat([eval_train_df, eval_valid_df]).sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

print(f"Evaluating {len(eval_users):,} users (sampled ranking: 1 positive + 100 negatives)...")

sequences_df = recommender._build_sequences(eval_history_df)
user_order = sequences_df["USER_ID"].tolist()

all_scores = recommender.scorer.estimator.predict_proba_with_embeddings(interactions=sequences_df)
item_name_to_idx = {name: i for i, name in enumerate(scorer.item_names)}

gt_lookup = eval_test_df.set_index("USER_ID")["ITEM_ID"].to_dict()
user_items = interactions.groupby("USER_ID")["ITEM_ID"].apply(set).to_dict()

TOP_K = 10
N_NEGATIVES = 100

hits, ndcgs = [], []
for i, user_id in enumerate(user_order):
    test_item = gt_lookup.get(user_id)
    if test_item is None:
        continue
    seen = user_items.get(user_id, set())
    candidates = all_item_ids[~np.isin(all_item_ids, list(seen))]
    neg_sample = rng.choice(candidates, size=min(N_NEGATIVES, len(candidates)), replace=False)
    candidate_ids = [test_item] + list(neg_sample)
    candidate_idxs = [item_name_to_idx[c] for c in candidate_ids if c in item_name_to_idx]
    candidate_scores = all_scores[i, candidate_idxs]
    test_score = all_scores[i, item_name_to_idx[test_item]]
    rank = int((candidate_scores > test_score).sum()) + 1
    if rank <= TOP_K:
        hits.append(1)
        ndcgs.append(1.0 / np.log2(rank + 1))
    else:
        hits.append(0)
        ndcgs.append(0.0)

print(f"\n{'=' * 40}")
print(f"Evaluation: 1 positive + {N_NEGATIVES} random negatives")
print(f"HR@{TOP_K}   : {np.mean(hits):.4f}")
print(f"NDCG@{TOP_K} : {np.mean(ndcgs):.4f}")
print(f"Users evaluated: {len(hits):,}")
print(f"{'=' * 40}")

Evaluating 6,040 users (sampled ranking: 1 positive + 100 negatives)...


2026-04-29 01:28:12,214 - skrec.recommender.sequential.sequential_recommender - INFO Built sequences for 6040 users (max_len=200, has_outcome=True).


2026-04-29 01:28:12,214 skrec.recommender.sequential.sequential_recommender INFO Built sequences for 6040 users (max_len=200, has_outcome=True).



Evaluation: 1 positive + 100 random negatives
HR@10   : 0.8164
NDCG@10 : 0.5548
Users evaluated: 6,040


## 7. HR@10 Breakdown by Test Item Rating

In [7]:
gt_rating_lookup = eval_test_df.set_index("USER_ID")["OUTCOME"].to_dict()

records = []
for i, user_id in enumerate(user_order):
    test_item = gt_lookup.get(user_id)
    test_rating = gt_rating_lookup.get(user_id)
    if test_item is None:
        continue
    seen = user_items.get(user_id, set())
    candidates = all_item_ids[~np.isin(all_item_ids, list(seen))]
    neg_sample = rng.choice(candidates, size=min(N_NEGATIVES, len(candidates)), replace=False)
    candidate_ids = [test_item] + list(neg_sample)
    candidate_idxs = [item_name_to_idx[c] for c in candidate_ids if c in item_name_to_idx]
    candidate_scores = all_scores[i, candidate_idxs]
    test_score = all_scores[i, item_name_to_idx[test_item]]
    rank = int((candidate_scores > test_score).sum()) + 1
    hit = int(rank <= TOP_K)
    ndcg = (1.0 / np.log2(rank + 1)) if hit else 0.0
    records.append({"test_rating": int(test_rating), "hit": hit, "ndcg": ndcg})

breakdown = (
    pd.DataFrame(records)
    .groupby("test_rating")
    .agg(
        n_users=("hit", "count"),
        HR10=("hit", "mean"),
        NDCG10=("ndcg", "mean"),
    )
    .round(4)
)

print("HR@10 and NDCG@10 by test item rating (no explicit negatives, sampled eval):")
print(breakdown.to_string())

HR@10 and NDCG@10 by test item rating (no explicit negatives, sampled eval):
             n_users    HR10  NDCG10
test_rating                         
1                407  0.7224  0.3963
2                657  0.7656  0.4642
3               1413  0.8139  0.5397
4               1990  0.8302  0.5888
5               1573  0.8455  0.6091


## 8. Sample Recommendations

In [8]:
movie_title = movies.set_index(movies["MovieID"].astype(str))["Title"].to_dict()
gt_rating_lookup = eval_test_df.set_index("USER_ID")["OUTCOME"].to_dict()

# Use train+valid history (same as evaluation) so sequences are consistent
top_k_recs = recommender.recommend(interactions=eval_history_df, top_k=TOP_K)

for user_id in user_order[:5]:
    idx = user_order.index(user_id)
    recs = list(top_k_recs[idx])
    test_item = gt_lookup.get(user_id, "?")
    test_rating = gt_rating_lookup.get(user_id, "?")
    hit = "HIT" if test_item in recs else "MISS"
    print(f"\nUser {user_id}  |  Test: {movie_title.get(test_item, test_item)} (rated {test_rating:.0f}/5)  [{hit}]")
    print("  Top-10 (full-item ranking):")
    for rank, item_id in enumerate(recs, 1):
        marker = " <-- TEST ITEM" if item_id == test_item else ""
        print(f"    {rank:2}. {movie_title.get(item_id, item_id)}{marker}")

2026-04-29 01:28:15,765 - skrec.recommender.sequential.sequential_recommender - INFO Built sequences for 6040 users (max_len=200, has_outcome=True).


2026-04-29 01:28:15,765 skrec.recommender.sequential.sequential_recommender INFO Built sequences for 6040 users (max_len=200, has_outcome=True).



User 1  |  Test: Pocahontas (1995) (rated 5/5)  [MISS]
  Top-10 (full-item ranking):
     1. Mulan (1998)
     2. Lion King, The (1994)
     3. Aladdin (1992)
     4. Bug's Life, A (1998)
     5. Beauty and the Beast (1991)
     6. Tarzan (1999)
     7. Fantasia 2000 (1999)
     8. Muppet Christmas Carol, The (1992)
     9. Little Princess, A (1995)
    10. James and the Giant Peach (1996)

User 10  |  Test: Hero (1992) (rated 5/5)  [MISS]
  Top-10 (full-item ranking):
     1. To Kill a Mockingbird (1962)
     2. Rear Window (1954)
     3. Vertigo (1958)
     4. 12 Angry Men (1957)
     5. Grapes of Wrath, The (1940)
     6. Streetcar Named Desire, A (1951)
     7. Wizard of Oz, The (1939)
     8. It's a Wonderful Life (1946)
     9. Some Like It Hot (1959)
    10. Amadeus (1984)

User 100  |  Test: Apocalypse Now (1979) (rated 2/5)  [MISS]
  Top-10 (full-item ranking):
     1. Sling Blade (1996)
     2. Good Will Hunting (1997)
     3. Shawshank Redemption, The (1994)
     4. October